# Extract burst fraction and size data from manually scored FISH images

- Before using this notebook, images are scored using the "Cell counter" plugin in Fiji to identify "Transcription spot" foci, as well as "Nuclear control" and "External control" regions inside and outside the cell respectively. Cells without transcription spots are marked as having "No spots" to allow for measurement of the burst fraction, while objects to ignore are stored as "Ignore cell". XY coordinates are stored for each image as XY_[your-image-name].csv as well as the counts of the annotated points as Counts_[your-image-name].csv. For this analysis, randomised file names are used so the notebook includes de-randomisation of these image names. The notebook measures spot intensity from the images using the manually annotated coordinates, corrects for nuclear background, and calculates burst fraction, for simple estimates of nascent RNA output. Images with both RNA FISH and immunofluorescence channels can also be used and meta-images of the different channels are generated.

- Use the settings.yaml file to input experiment metadata, microscopy settings, file paths, and other necessary parameters. Avoid editing settings marked with "do not edit" as these will be automatically populated from the notebook.

## Prepare datasets

### Packages and files

#### Import packages

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
mpl.rcParams['figure.dpi'] = 300
import glob
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.ndimage import find_objects
import napari
from cellpose import io
%gui qt # qt app must be running in order to use napari
import re
from ruamel.yaml import YAML

# Set up YAML parser
yaml = YAML() 
yaml.preserve_quotes = True
yaml.default_flow_style = False

### Configuration settings file – should be stored alongside your notebooks ###
config_path = "path/to/settings.yaml"

#### Experiment-specific parameters

In [ ]:
with open(config_path, 'r') as f:
    config = yaml.load(f)

# Extract experiment settings
dateNoTime = config['experiment']['date']
dateFormat = str(config['experiment']['dateFormat'])  # Convert to str to avoid YAML type issues with pandas

# Extract microscopy settings
ZYXscalar = config['microscopy']['ZYXscalar']
channelNames = config['microscopy']['channelNames']

# Extract analysis parameters
fishSuffix = config['analysis']['fishSuffix']
nucChan = config['analysis']['nucChan']
cellChan = config['analysis']['cellChan']

# Extract and construct file paths
tiffPath = config['paths']['tiffPath']
dataPath = config['paths']['dataPath']

resultsPath = config['paths']['outputFolderPath']
pathCheck = Path(resultsPath)
if not pathCheck.exists():
    pathCheck.mkdir(parents=True, exist_ok=True)  

print(f"Configuration loaded from {config_path}")
print(f"Experiment date: {dateNoTime}")
print(f"TIFF path: {tiffPath}")
print(f"Data path: {dataPath}")
print(f"Results path: {resultsPath}")

#### Read in images and analysis files

In [ ]:
# get file names of "Counts____.csv" and "XY____.csv" files
ctStart = '/Counts*'
ctFileType = '*.csv'
ctPattern = ''.join([ctStart, ctFileType])
ctFiles = sorted(glob.glob(dataPath + ctPattern))
print(f"Number of Counts files =",len(ctFiles))

xyStart = '/XY*'
xyFileType = '*.csv'
xyPattern = ''.join([xyStart, xyFileType])
xyFiles = sorted(glob.glob(dataPath + xyPattern))
print(f"Number of XY files =",len(xyFiles))

# Save to YAML file for later use
# Load settings from YAML configuration file
with open(config_path, 'r') as f:
    config = yaml.load(f)
    
config['paths']['ctFilesPath'] = ctFiles
config['paths']['xyFilesPath'] = xyFiles

with open(config_path, 'w') as f:
    yaml.dump(config, f)

# read in the random number list and original names
randPath = tiffPath + '/randNumList.txt'
randOrder = pd.read_csv(randPath,sep="\t",header=None)
randOrder.columns = ["randOrder"]
print(f"Length of random number list: {len(randOrder)}")

origPath = tiffPath + '/originalOrder.txt'
origImNames = pd.read_csv(origPath,sep="\t",header=None,encoding='latin1')
print(f"Number of images in list: {len(origImNames)}")
###

#### Define measurement parameters if not using default

In [ ]:
############### MEASUREMENT PARAMETERS
## Spot measurement

# Default settings – 0.5 µm radius and 0.65µm height (based on typical size of transcription foci)
spotRadius = np.round((0.5/ZYXscalar[1])/2)
spotHeight = np.round(0.65/ZYXscalar[0])

# Manually define (n pixels)
# spotRadius = 2
# spotHeight = 5

## Metaplot

# Default settings – 2 µm radius and 2µm height
metaRadius = np.round((2/ZYXscalar[1])/2)
metaHeight = spotHeight

# Manually define (n pixels)
# metaRadius = 15
# metaHeight = spotHeight

###############

print(f"Spot measurement cylinder is {np.round((spotRadius*2+1)*ZYXscalar[1],2)}x{np.round((spotRadius*2+1)*ZYXscalar[1],2)}x{np.round(spotHeight*ZYXscalar[0],2)}µm in size, which is {int(spotRadius*2)+1} x {int(spotRadius*2)+1} x {int(spotHeight)} pixels")
print(f"Meta image size is {np.round((metaRadius*2+1)*ZYXscalar[1],2)}x{np.round((metaRadius*2+1)*ZYXscalar[1],2)}x{np.round(metaHeight*ZYXscalar[0],2)}µm in size, which is {int(metaRadius*2)+1} x {int(metaRadius*2)+1} x {int(metaHeight)} pixels")
print("\n")

# Save measurement parameters to config file
config_path = "path/to/settings.yaml"
with open(config_path, 'r') as f:
    config = yaml.load(f)

config['analysis']['spotRadius'] = int(spotRadius)
config['analysis']['spotHeight'] = int(spotHeight)
config['analysis']['metaRadius'] = int(metaRadius)
config['analysis']['metaHeight'] = int(metaHeight)

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print(f"Measurement parameters saved to {config_path}")

### Collect image metadata

#### Metadata function

In [ ]:
### ADD IN OPTIONS FOR METADATA PARAMETERS IN FILENAMES BELOW ###
### IDEALLY FILES SHOULD BE NAMED WITH UNDERSCORES BETWEEN METADATA PARAMETERS ###
###  e.g. 2023-08-06_WT_A1_SOX2_mESC_DTAG_d1.tif ###

def filenameMetadata(origFileName,randFileName): 

    # split by _ *** MUST HAVE UNDERSCORE SEPARATORS IN FILENAME ***
    splits = origFileName.split("_")

    # set up criteria lists to extract metadata from file name – do not leave blank! Delete as appropriate
    genotype = [""] # e.g. ["WT"]
    
    clone = [""] # e.g. ["A2","C5"]
    
    probes = [""] # e.g.['MYC','SOX2']

    cellType = [""] # e.g.["mESC"]

    selection = [""] # e.g.["DMSO","DTAG"] or ["d1","d2","d3"] or ["Day1","Day2","Day3"]

    # remove parameters as appropriate here
    sample_dict = {
        'genotype': None,
        'imageDate': None,
        'clone': None,
        'probeCombo': None,
        'cellType': None,
        'selection': None
    }
    sample_dict['imageDate'] = pd.to_datetime(splits[0],format=dateFormat)

    # extract metadata
    probe_combo = []
    for s in splits[1:]:
        if bool(re.search(r'\b[Dd]\d',s)): # look for instances of "d" with a number attached indicating day of differentiation/expansion
            sample_dict['day']  = s #int("".join([i for i in s if i.isdigit()]))
        elif bool(re.search(r'\b[Dd]ay\d',s)): # look for instances of "day" with a number attached indicating day of differentiation/expansion
            sample_dict['day']  = s #int("".join([i for i in s if i.isdigit()]))
        elif [sel for sel in selection if sel.lower() in s.lower()]:
            sample_dict['selection'] = s
        elif [gen for gen in genotype if re.search(gen,s)]:
            sample_dict['genotype']  = s
        elif [pc for pc in probes if pc.lower() in s.lower()]:
            probe_combo.append(s)
        elif s.upper() in cellType:
            sample_dict['cellType'] = s
        elif s.upper() in clone:
            sample_dict['clone'] = s 
    probe_combo = " ".join(probe_combo)
    sample_dict['probeCombo'] = probe_combo

    sample_dict['randomisedName'] = randFileName
    randNum = re.findall(r'\d{3}',randFileName)
    sample_dict['randomisedID'] = int(randNum[0])

    # collect sample metadata together
    annotData = pd.Series(sample_dict)

    return annotData

#### Extract metadata

In [ ]:
# Extract metadata from the randomised image names and store in a data frame

def derandomiseMetadata(file,randomOrderDF):
    # get name of file without path
    fileOnly = file.split('/')
    fileOnly = fileOnly[-1]

    # get random file name/number
    randFileName = re.findall(r'image\d{3}(?=\D)',fileOnly)
    if len(randFileName) == 1:
        randFileName = randFileName[0] # should only be one instance
    else:
        print(f"Multiple potential file names found with image: {fileOnly}")

    randFileNum = re.findall(r'\d{3}',randFileName)
    randFileNum = int(randFileNum[0])

    # find the index of that number in the random number list
    mask = randomOrderDF["randOrder"] == randFileNum
    numID = randomOrderDF[mask]
    origNum = int(numID.index.values[0])

    # extract the original image name using this index
    origFileName = origImNames.iloc[origNum,0]

    # create metadata 
    fileMetadata = filenameMetadata(origFileName,randFileName)

    return fileMetadata

fileMetaList = [derandomiseMetadata(file,randOrder) for file in xyFiles]

metadata_df = pd.DataFrame(fileMetaList)
metaColumns = metadata_df.columns
metadata_df

#### Choose appropriate metadata columns for plotting

In [ ]:
# Define which metadata columns to include in downstream dataframes. 
# This will determine how the data can be stratified during plotting
# E.g. if you have samples split by genotype include "genotype", if you have
# different stages of differentiation include "selection" or "day" etc. 
metadata_for_plotting = ["selection", "clone","acqImNum"]  # Edit this list as appropriate for your dataset

# Available metadata columns to choose from:
print("Available metadata columns:", metadata_df.columns.tolist())
print(f"Chosen metadata columns for downstream analysis: {metadata_for_plotting}")

# Save metadata plotting parameters to config file
config_path = "path/to/settings.yaml"
with open(config_path, 'r') as f:
    config = yaml.load(f)

config['analysis']['metadata_for_plotting'] = metadata_for_plotting

with open(config_path, 'w') as f:
    yaml.dump(config, f)

### Extract count and XY data from files

In [ ]:
def extractCountData(index,ctFile):
    ctData = pd.read_csv(ctFile,sep=',')

    # define and find columns to extract – edit names if more necessary columns
    colsToExtract = ["Transcription spot","Nuclear control","External control","No spots","Ignore cell"] ### edit if more necessary columns
    
    colNums = [ctData.columns.get_loc(col) for col in colsToExtract]

    # extract total count row and relevant columns and store in list
    ctData = ctData.iloc[-1,colNums] 
    ctData["imageNum"] = index

    return ctData 

def extractXYData(index,xyFile):
    xyData = pd.read_csv(xyFile,sep=',')

    # XY coordinates require adjustment (zero-based from Cell Counter)
    xyData["X"] = xyData["X"]+1
    xyData["Y"] = xyData["Y"]+1

    xyData["imageNum"] = index
    
    return xyData

def getXY_metadata(index,xyDF,metadataDF):
    # how many rows for each image?
    nRows = len(xyDF[xyDF["imageNum"] == index])
    
    # create dataframe with the same metadata information for each row of xy points
    xy_metadata = pd.concat([metadataDF.loc[index,:]] * nRows,axis=1).transpose().reset_index(drop=True)
    
    return xy_metadata

# check listed files have same names
for xy,ct in zip(xyFiles,ctFiles):
    xyFile = xy.split('/')[-1]
    xyEnd = xyFile.split('_')[-1]
    ctFile = ct.split('/')[-1]
    ctEnd = ctFile.split('_')[-1]

    if xyEnd != ctEnd:
        print("Warning: potential naming error among files")    
    print(xyFile,ctFile)

# extract necessary count data from each file
countList = [extractCountData(index,ctFile) for index,ctFile in enumerate(ctFiles)]
ct_df = pd.DataFrame(countList).reset_index(drop=True)
ct_df = pd.concat([ct_df,metadata_df],axis=1)

# extract necessary XY data from each file
xyList = [extractXYData(index,xyFile) for index,xyFile in enumerate(xyFiles)]
xy_df = pd.concat(xyList).reset_index(drop=True)

# convert separate XYZ values into a single tuple from simpler use later
def getXYcentroid(row):
    return tuple([row["Z-pos"],row["Y"],row["X"]])

xy_df["centroid"] = xy_df.apply(getXYcentroid,axis=1)

# attach metadata from images to each row of xy data
metadataList = [getXY_metadata(index,xy_df,metadata_df) for index,xyFile in enumerate(xyFiles)]
xyMetadata = pd.concat(metadataList).reset_index(drop=True)
xyMetadata

xy_df = pd.concat([xy_df,xyMetadata],axis=1).reset_index(drop=True)
xy_df

### Link transcription spots to nearest controls

In [ ]:
##### FUNCTIONS
# Define a function to extract and scale centroid ZYX coordinates and store together in filtered dataframe
def extract_scaledCentCoords(row):
    column_value = [row['Z-pos'],row['Y'],row['X']]
    scaled_coords = np.array(column_value) * ZYXscalar - ZYXscalar
    scaled_coords = np.round(scaled_coords,2)
    return tuple(scaled_coords)

xy_df.loc[:,"scaledCentroid"] = xy_df.apply(extract_scaledCentCoords, axis=1)

def origScaleCentroid(coordinates):
    rescaleCentroid = np.asarray(coordinates)/ZYXscalar
    return tuple(np.round(rescaleCentroid,1).astype(int))

def closestCoord(point,pointList):
    ### find the closest coordinate in a list of coordinates by euclidean distance
    # find distance to all points
    distList = cdist([point],pointList,'euclidean')
    # get ID and value of shortest distance
    shortDistID = np.argmin(distList)
    shortestDist = np.amin(distList)
    # identify which coordinate is closest
    centroid = tuple(np.round(np.asarray(pointList[shortDistID])/ZYXscalar,1).astype(int))
    return centroid,shortestDist,shortDistID

def ncCloseCentroid(row):
    return closestCoord(row["scaledCentroid"],ncCentroids)[0]

def ncCloseDistance(row):
    return closestCoord(row["scaledCentroid"],ncCentroids)[1]

def ncCloseID(row):
    return closestCoord(row["scaledCentroid"],ncCentroids)[2]

def ecCloseCentroid(row):
    return closestCoord(row["scaledCentroid"],ecCentroids)[0]

def ecCloseDistance(row):
    return closestCoord(row["scaledCentroid"],ecCentroids)[1]

def ecCloseID(row):
    return closestCoord(row["scaledCentroid"],ecCentroids)[2]

#####

# Set up list of lists for data collection
# Combine fixed columns with user-defined metadata columns
fixed_cols = ["targetSiteID","targetSiteCentroid","nucleusControlID","nucleusCentroid",
              "shortestDistance_nuclear","externalControlID","externalCentroid","shortestDistance_external"]
colNames = ["imageNum"] + metadata_for_plotting + fixed_cols 
dataLists = {col: [] for col in colNames}

for index, temp_df in xy_df.groupby(["imageNum"]):
    
    image_num = index[0]
    # print(f"Image: {image_num}")

    # if all values are type 4 ("No spots") then don't do distances calculation, otherwise carry on
    # need to remove "Ignore cell" category here to avoid confusion
    typeVals = temp_df["Type"].values
    typeVals = typeVals[typeVals != 5]

    if len(typeVals) == 0: # if all type 5 ignore
        continue
    elif (len(np.unique(typeVals)) == 1) & (typeVals[0] == 4): # if all type 4 ignore
        continue
    else: 
        # transcription sites
        targetTemp_df = temp_df[temp_df["Type"] == 1].copy()
        # nuclear controls
        ncTemp_df = temp_df[temp_df["Type"] == 2].copy()
        ncCentroids = ncTemp_df["scaledCentroid"].tolist()
        # external controls
        ecTemp_df = temp_df[temp_df["Type"] == 3].copy()
        ecCentroids = ecTemp_df["scaledCentroid"].tolist()

        # Validation check for mismatched numbers of target sites and nuclear controls
        if len(targetTemp_df) != len(ncTemp_df):
            print(f"WARNING: Image {image_num} has {len(targetTemp_df)} target site(s) but {len(ncTemp_df)} nuclear control(s)")
            print(f"Check image annotation file on Fiji for mis-assigned markers and amend if necessary.")
            print(f"The image filename is: image{image_num+1:03d}.tif.")
            print(f"You will need to edit the annotations, re-export the files, then re-run the analysis to this point.\n")
            continue

        # Validation check for missing external controls
        if len(targetTemp_df) > 0 and len(ncTemp_df) > 0 and len(ecTemp_df) == 0:
            print(f"WARNING: Image {image_num} has {len(targetTemp_df)} target site(s) and {len(ncTemp_df)} nuclear control(s) but NO external controls!")
            print(f"Check image annotation for missing external control markers.")
            print(f"The image filename is: image{image_num+1:03d}.tif.") 
            print(f"You will need to edit the annotations, re-export the files, then re-run the analysis to this point.\n")
            continue

        # take relevant data from original data frame to add to new dataframe where each target site is matched 
        # with its closest controls
        dataLists["imageNum"].extend(targetTemp_df["imageNum"].tolist())
        # add user-defined metadata columns
        for metadata_col in metadata_for_plotting:
            dataLists[metadata_col].extend(targetTemp_df[metadata_col].tolist())
        # print(f"Image {image_num}: section 1")

        # target sites – IDs, centroids 
        dataLists["targetSiteID"].extend(targetTemp_df.index.values)
        dataLists["targetSiteCentroid"].extend(targetTemp_df["scaledCentroid"].apply(origScaleCentroid).tolist())
        # print(f"Image {image_num}: section 2")
        
        # nuclear controls – IDs, centroid, distance to target site
        ncIDs = ncTemp_df.index.values[targetTemp_df.apply(ncCloseID,axis=1).tolist()]
        dataLists["nucleusControlID"].extend(ncIDs)
        dataLists["nucleusCentroid"].extend(targetTemp_df.apply(ncCloseCentroid,axis=1).tolist())
        dataLists["shortestDistance_nuclear"].extend(targetTemp_df.apply(ncCloseDistance,axis=1).tolist())
        # print(f"Image {image_num}: section 3")

        # external controls – IDs, centroid, distance to target site
        ecIDs = ecTemp_df.index.values[targetTemp_df.apply(ecCloseID,axis=1).tolist()]
        dataLists["externalControlID"].extend(ecIDs)
        dataLists["externalCentroid"].extend(targetTemp_df.apply(ecCloseCentroid,axis=1).tolist())
        dataLists["shortestDistance_external"].extend(targetTemp_df.apply(ecCloseDistance,axis=1).tolist())
        # print(f"Image {image_num}: section 4\n")

targetToNcEc = pd.DataFrame(dataLists)
targetToNcEc.loc[0:9,:]

#### Find image files and import

In [ ]:
imPath = tiffPath + '/randomisedFiles'

imStart = '/*'
imFileType= '*.tif'
imPattern = ''.join([imStart, imFileType])
imFiles = sorted(glob.glob(imPath + imPattern))
print(f"Number of image files =",len(imFiles))

if not len(imFiles) == len(xyFiles):
    print("*** Number of images does not match number of files! ***")
    print(f"Number of XY files =",len(xyFiles))

#### Plot coordinates onto images – sanity check

In [ ]:
imNum = 3

img = io.imread(imFiles[imNum])
print(img.shape)

channelDim = 1

# if no target sites, then adjust image number to ensure target sites found
print(f"\nNumber of linked target sites: {len(targetToNcEc[targetToNcEc['imageNum'] == imNum])}")
print("If number of target sites is 0, try a different image number (imNum)")

# check centroids/images in Napari to confirm they match up before proceeding with measurements
print("Click on the Napari viewer to check centroids match up with image before proceeding with measurements.")
print("Use the z-slice slider to navigate through the image and confirm that the coloured foci match with expected transcription foci positions")

# select FISH channel
cellImg = img[:,cellChan,:,:]
nucImg = img[:,nucChan,:,:]

# extract centroids
imBool = (targetToNcEc["imageNum"] == imNum)
targetCentroids = np.vstack(targetToNcEc.loc[imBool,"targetSiteCentroid"])
ncCentroids = np.vstack(targetToNcEc.loc[imBool,"nucleusCentroid"])
ecCentroids = np.vstack(targetToNcEc.loc[imBool,"externalCentroid"])

# plot using napari
viewer = napari.view_image(cellImg)
nucImg = viewer.add_image(nucImg,opacity=0.5,colormap="bop blue")
targetSites = viewer.add_points(targetCentroids,size=10,face_color='red',opacity=0.5)
nucCont = viewer.add_points(ncCentroids,size=10,face_color='cyan',opacity=0.5)
extCont = viewer.add_points(ecCentroids,size=10,face_color='yellow',opacity=0.5)
napari.run()

## MEASURE SPOT INT: Create mask for spots and measure

Check image shape and define which channel is which

In [ ]:
img = io.imread(imFiles[0])
print(f"Image shape: {img.shape}")

Confirm that the dimensions of the slices and channels in the "Image shape:" above are as expected. Change below if not.

In [ ]:
### Check slice and channel dimensions are correct! Remember that this is zero-based.
origSliceDim = 0
origChannelDim = 1

### Measure mean spot intensity and cropped multi-channel image

In [ ]:
# create cropped image around centroid of defined size
def cropMetaImage(image,distanceMap,centroid,imageRadius,imageHeight):
    
    # create mask for region around centroid to crop
    metaMask = (distanceMap <= imageRadius) & (z >= centroid[0] - imageHeight/2) & (z <= centroid[0] + imageHeight/2)
    
    # get coordinates
    bbMeta = find_objects(metaMask)
    z_slice, row_slice, col_slice = bbMeta[0]

    # crop image and make max projection
    imCrop = image[z_slice.start:z_slice.stop, row_slice.start:row_slice.stop, col_slice.start:col_slice.stop]
    imCropMax = np.max(imCrop,0)
    
    return imCropMax
    
# get both mean spot intensity of pixels within cylinder around centroid, and cropped image of this region
def measureMeanSpotInt(image,meshgrid,centroid,measureRadius,measureHeight):

    if type(centroid) == "tuple":
        centroid = np.asarray(centroid)

    # Create distance map from point of interest to all other points in mesh grid (2D)
    distMap = np.sqrt((meshgrid[1] - centroid[1]) ** 2 + (meshgrid[2] - centroid[2]) ** 2)

    # Make mask binary based on distance to centroid and z-position
    spotMask = (distMap <= measureRadius) & (meshgrid[0] >= centroid[0] - measureHeight/2) & (meshgrid[0] <= centroid[0] + measureHeight/2)

    # Measure intensities – either from single image or same point in multiple images
    if len(image.shape) > 3:
        # Calculate intensity within spot region of all channels provided
        meanSpotInt = [np.mean(chan[spotMask]) for chan in np.rollaxis(image,channelDim)]
        # Create cropped image around target centroid in each channel for meta analysis
        croppedSpot = [cropMetaImage(chan,distMap,centroid,metaRadius,metaHeight) for chan in np.rollaxis(image,channelDim)]

        return meanSpotInt, croppedSpot
    else:        
        # Take mean of values within the spot measurement mask
        meanSpotInt = np.mean(image[spotMask])
        croppedSpot = cropMetaImage(image, distMap, centroid, metaRadius, metaHeight)

        return meanSpotInt, croppedSpot

# use above function on dataframe by row
###
def targetSpotAllChannelsByRow(row):
    return measureMeanSpotInt(img,meshGrid,row["targetSiteCentroid"],spotRadius,spotHeight)

def ncSpotAllChannelsByRow(row):
    return measureMeanSpotInt(img,meshGrid,row["nucleusCentroid"],spotRadius,spotHeight)

def ecSpotAllChannelsByRow(row):
    return measureMeanSpotInt(img,meshGrid,row["externalCentroid"],spotRadius,spotHeight)

# set up list of lists for data collectio

# spotCols = ["imageNum","genotype","selection","targetCentroid","targetSiteIntAllChan","targetSiteCropAllChan","ncCentroid","nuclearControlIntAllChan","nuclearControlCropAllChan","ecCentroid","externalControlIntAllChan","externalControlCropAllChan"]

fixed_cols = ["targetCentroid","targetSiteIntAllChan","targetSiteCropAllChan","ncCentroid",
              "nuclearControlIntAllChan","nuclearControlCropAllChan","ecCentroid","externalControlIntAllChan","externalControlCropAllChan"]
spotCols = ["imageNum"] + metadata_for_plotting + fixed_cols 
dataLists = {col: [] for col in spotCols}

for index, temp_df in targetToNcEc.groupby(["imageNum"]):
    
    image_num = index[0]
    print(f"Processing image {image_num}")

    # check files are equivalent and if so load images
    randImBase = (imFiles[0].split('/')[-1]).split('.')[0] # get the part with randomised name 'imageXXX' 
    randXYBase = (xyFiles[0].split('_')[-1]).split('.')[0]
    if randImBase != randXYBase:
        print('Error: XY and image file lists are not properly aligned')
        break
    else:
        img = io.imread(imFiles[image_num])

    # alter channel and slice dimensions to fit downstream requirements, slice in 1st dimension, channel in second
    print(f"Original image shape: {img.shape}")
    if len(img.shape) == 4:
        if origChannelDim != 1:
            img = np.moveaxis(img,origChannelDim,1)
            channelDim = 1
        else: 
            channelDim = origChannelDim

        if origSliceDim != 0:
            img = np.moveaxis(img,origSliceDim,0)
            sliceDim = 0
        else: 
            sliceDim = origSliceDim

        # perhaps don't need to make separate variables for these but marginally easier to read            
        imgDAPI = img[:,nucChan,:,:]
        imgFISH = img[:,cellChan,:,:]

    print(f"Analysed image shape: {img.shape}")

    # Create meshgrid same size as the image – uses xy Cartesian coordinates, not ij
    print("Creating meshgrid...")
    y, z, x = np.meshgrid(np.arange(imgFISH.shape[1]), np.arange(imgFISH.shape[0]), np.arange(imgFISH.shape[2]))

    meshGrid = list([z,y,x])

    dataLists["imageNum"].extend(temp_df["imageNum"].copy().tolist())
    # add user-defined metadata columns
    for metadata_col in metadata_for_plotting:
        dataLists[metadata_col].extend(temp_df[metadata_col].tolist())
        
    dataLists["targetCentroid"].extend(temp_df["targetSiteCentroid"].copy().tolist())
    print("Measuring target foci intensities...")
    targetSpotTempResults = temp_df.apply(targetSpotAllChannelsByRow, axis=1)
    dataLists["targetSiteIntAllChan"].extend(targetSpotTempResults.apply(lambda x: x[0]).tolist())
    dataLists["targetSiteCropAllChan"].extend(targetSpotTempResults.apply(lambda x: x[1]).tolist())
    dataLists["ncCentroid"].extend(temp_df["nucleusCentroid"].copy().tolist())
    print("Measuring nuclear control intensities...")
    ncSpotTempResults = temp_df.apply(ncSpotAllChannelsByRow, axis=1)
    dataLists["nuclearControlIntAllChan"].extend(ncSpotTempResults.apply(lambda x: x[0]).tolist())
    dataLists["nuclearControlCropAllChan"].extend(ncSpotTempResults.apply(lambda x: x[1]).tolist())
    dataLists["ecCentroid"].extend(temp_df["externalCentroid"].copy().tolist())
    print("Measuring external control intensities...\n")
    ecSpotTempResults = temp_df.apply(ecSpotAllChannelsByRow, axis=1)
    dataLists["externalControlIntAllChan"].extend(ecSpotTempResults.apply(lambda x: x[0]).tolist())
    dataLists["externalControlCropAllChan"].extend(ecSpotTempResults.apply(lambda x: x[1]).tolist())

# assemble lists into dataframe
spotData = pd.DataFrame(dataLists,columns=spotCols)

# split spot intensity data from multiple channels into separate columns
def splitMultiChanMeasure(df,colToSplit,newColRoot,channelNames):
    # split column of multiple data points into individual columns
    newCols = df[colToSplit].apply(pd.Series)

    # rename new columns with channel suffixes
    newCols.columns = [f"{newColRoot}_{suffix}" for suffix in channelNames]

    # get the position of the original column and drop this column
    newColPos = spotData.columns.get_loc(colToSplit)
    df.drop(columns=[colToSplit], inplace=True)
    
    # insert new columns at the original column position
    for index, col in enumerate(newCols.columns):
        df.insert(newColPos + index, col, newCols[col])
    
    return df

spotData = splitMultiChanMeasure(spotData,"targetSiteIntAllChan","targetSiteInt",channelNames)
spotData = splitMultiChanMeasure(spotData,"nuclearControlIntAllChan","nuclearControlInt",channelNames)
spotData = splitMultiChanMeasure(spotData,"externalControlIntAllChan","externalControlInt",channelNames)
spotData = splitMultiChanMeasure(spotData,"targetSiteCropAllChan","targetSiteCrop",channelNames)
spotData = splitMultiChanMeasure(spotData,"nuclearControlCropAllChan","nuclearControlCrop",channelNames)
spotData = splitMultiChanMeasure(spotData,"externalControlCropAllChan","externalControlCrop",channelNames)

# normalise intensities to background
def normaliseIntensities(row):
    row["targetNormalised_nc"] = row[f"targetSiteInt_{fishSuffix}"] - row[f"nuclearControlInt_{fishSuffix}"]
    row["targetNormalised_ec"] = row[f"targetSiteInt_{fishSuffix}"] - row[f"externalControlInt_{fishSuffix}"]
    row["targetNormalised_nc-ecRatio"] = (row[f"targetSiteInt_{fishSuffix}"] - row[f"externalControlInt_{fishSuffix}"])/(row[f"nuclearControlInt_{fishSuffix}"] - row[f"externalControlInt_{fishSuffix}"])
    return row

spotData = spotData.apply(normaliseIntensities,axis=1)
spotData

Reorder dataframe by desired parameters – edit with your list of genotypes

In [ ]:
# sort genotypes by custom ordering
def categoricalColumn(df,columnName):
    catOrder = [""] # e.g. ["WT","mutant1"]
    df[f'{columnName}Cat'] = pd.Categorical(df[columnName], categories=catOrder, ordered=True)
    return df

# sort data by genotype and selection – if only using one parameter then edit spotData.sort_values(by=["parameter"]) etc.
columnName = "" # e.g. "genotype"
spotData = categoricalColumn(spotData,columnName)
spotData = spotData.sort_values(by=[f"{columnName}Cat"]).reset_index(drop=True)

### Save spot data 

In [ ]:
# use pickle for spotData as cropped images are not corrected read when re-importing from .csv
spotDataPath = resultsPath + '/spotData_' + dateNoTime + '.pkl'
spotData.to_pickle(spotDataPath)

spotDataPath = resultsPath + '/spotData_' + dateNoTime + '.csv'
spotData.to_csv(spotDataPath,index=False)

xyDataPath = resultsPath + '/xyData_' + dateNoTime + '.csv'
xy_df.to_csv(xyDataPath,index=False)

countDataPath = resultsPath + '/countData_' + dateNoTime + '.csv'
ct_df.to_csv(countDataPath,index=False)

metadataPath = resultsPath + '/metadata_' + dateNoTime + '.csv'
metadata_df.to_csv(metadataPath,index=False)

Check that metaimages look reasonable – should see an enrichment of averaged foci in "Target spot" compared to "Nuclear control"

In [ ]:
def createMetaImage(dfCol):
    imArray = np.asarray(dfCol.tolist())
    metaIm = np.mean(imArray,0)
    return metaIm

# Check cropped images look okay
# spotNum = 2
viewer = napari.Viewer(title="Target spot")
cols = ["bop blue","yellow","green","magenta"]
for index,chan in enumerate(channelNames):
    viewer.add_image(createMetaImage(spotData[f"targetSiteCrop_{chan}"]),colormap=cols[index],name=f"{chan}",opacity=0.3)       
napari.run()

viewer2 = napari.Viewer(title="Nuclear control")
for index,chan in enumerate(channelNames):
    viewer2.add_image(createMetaImage(spotData[f"nuclearControlCrop_{chan}"]),colormap=cols[index],name=f"{chan}")       
napari.run()

### Collect summary statistics

In [ ]:
from functools import reduce

# ---------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------
keys = [""] # e.g. ["genotype","selection"] or ["genotype"] or ["selection"]
order = [""] # e.g. ["WT","KO","HET"] – edit with your list of genotypes in desired plotting order
meta_shape = (metaRadius * 2 + 1, metaRadius * 2 + 1)

# ---------------------------------------------------------------------
# Normalisation configuration
# Set these once for all relative calculations
# ---------------------------------------------------------------------
norm_col = ""       # e.g. "genotype" or "selection"
norm_value = ""           # or e.g. "control"
within_cols = None # None or e.g. ["genotype"]

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def set_ordered_category(df, col, order):
    df = df.copy()
    df[f"{col}Cat"] = pd.Categorical(df[col], categories=order, ordered=True)
    return df

def grouped_stat(df, group_cols, value_col, func, out_col):
    return (
        df.groupby(group_cols, dropna=False)[value_col]
          .agg(func)
          .reset_index(name=out_col)
    )

def relative_to_baseline(df, value_col, baseline_col, baseline_value, within_cols=None):
    """
    Compute value_col / baseline within each subgroup.

    Examples
    --------
    Normalize by genotype within each selection:
        relative_to_baseline(df, "spotFrequency", "genotype", "WT", ["selection"])

    Normalize by selection within each genotype:
        relative_to_baseline(df, "spotFrequency", "selection", "control", ["genotype"])
    """
    within_cols = [] if within_cols is None else list(within_cols)

    baseline_df = df.loc[df[baseline_col].eq(baseline_value), within_cols + [value_col]].copy()
    if baseline_df.empty:
        raise ValueError(
            f"No baseline rows found for {baseline_col} == {baseline_value!r} "
            f"within {within_cols if within_cols else 'the full dataset'}."
        )

    # Collapse duplicates safely if more than one baseline row exists per subgroup
    if within_cols:
        baseline_df = (
            baseline_df.groupby(within_cols, dropna=False, as_index=False)[value_col]
                       .mean()
                       .rename(columns={value_col: "_baseline"})
        )

        merged = df.merge(baseline_df, on=within_cols, how="left")
        out = merged[value_col] / merged["_baseline"]
        return out

    # Single baseline for the whole table
    baseline_value_num = baseline_df[value_col].mean()
    return df[value_col] / baseline_value_num

def create_meta_image(images, expected_shape=meta_shape):
    images = list(images)
    valid = []

    for i, im in enumerate(images):
        if getattr(im, "shape", None) != expected_shape:
            print(f"Image at index {i} has incorrect shape: {getattr(im, 'shape', None)}. Expected {expected_shape}")
        else:
            valid.append(im)

    if not valid:
        raise ValueError("No valid images to create a meta image.")

    return np.mean(np.stack(valid, axis=0), axis=0)

def meta_image_table(df, group_cols, image_col):
    return (
        df.groupby(group_cols, dropna=False)[image_col]
          .apply(create_meta_image)
          .reset_index(name=image_col)
    )

def merge_all(tables, on):
    return reduce(lambda left, right: left.merge(right, on=on, how="left"), tables)

# ---------------------------------------------------------------------
# Counts and frequencies
# ---------------------------------------------------------------------
counts = (
    ct_df.groupby(keys, dropna=False)
         .agg(
             nTargetSites=("Transcription spot", "sum"),
             externalControl=("External control", "sum"),
             nNoSpots=("No spots", "sum"),
         )
         .reset_index()
)

counts["nSpots"] = counts["externalControl"]
counts["nCells"] = counts["nSpots"] + counts["nNoSpots"]
counts["spotFrequency"] = counts["nSpots"] / counts["nCells"]
counts["meanSpotsPerCell"] = counts["nTargetSites"] / counts["nCells"]
counts.drop(columns=["externalControl"], inplace=True)

# ---------------------------------------------------------------------
# Intensity summaries
# ---------------------------------------------------------------------
intensity_map = {
    "targetNormalised_nc": "nucNorm",
    "targetNormalised_ec": "extNorm",
    "targetNormalised_nc-ecRatio": "nuc_extNorm",
}

summary = counts.copy()

for source_col, tag in intensity_map.items():
    mean_col = f"meanSpotInt_{tag}"
    median_col = f"medianSpotInt_{tag}"

    mean_df = grouped_stat(spotData, keys, source_col, "mean", mean_col)
    median_df = grouped_stat(spotData, keys, source_col, "median", median_col)

    summary = summary.merge(mean_df, on=keys, how="left")
    summary = summary.merge(median_df, on=keys, how="left")

# ---------------------------------------------------------------------
# Normalisation set-up
# ---------------------------------------------------------------------

relative_cols = {
    "spotFrequency": "relSpotFrequency",
    "meanSpotInt_nucNorm": "relMeanSpotInt_nucNorm",
    "meanSpotInt_extNorm": "relMeanSpotInt_extNorm",
    "meanSpotInt_nuc_extNorm": "relMeanSpotInt_nuc_extNorm",
    "medianSpotInt_nucNorm": "relMedSpotInt_nucNorm",
    "medianSpotInt_extNorm": "relMedSpotInt_extNorm",
    "medianSpotInt_nuc_extNorm": "relMedSpotInt_nuc_extNorm",
}

for value_col, out_col in relative_cols.items():
    summary[out_col] = relative_to_baseline(
        summary,
        value_col=value_col,
        baseline_col=norm_col,
        baseline_value=norm_value,
        within_cols=within_cols
    )

# ---------------------------------------------------------------------
# Meta images
# ---------------------------------------------------------------------

def generate_meta_columns(channel_names):
    return (
        [f"targetSiteCrop_{ch}" for ch in channel_names] +
        [f"nuclearControlCrop_{ch}" for ch in channel_names]
    )

meta_cols = generate_meta_columns(channelNames)

meta_tables = [
    meta_image_table(spotData, keys, col)
    for col in meta_cols
]

# ---------------------------------------------------------------------
# Final combined summary table
# ---------------------------------------------------------------------
summaryData = merge_all([summary, *meta_tables], on=keys)

summaryData = set_ordered_category(summaryData, norm_col, order)
summaryData = summaryData.sort_values(by=[f"{norm_col}Cat", "selection"]).reset_index(drop=True)

conditions_df = summaryData[keys].drop_duplicates().reset_index(drop=True)

summaryData

### Save summary data

In [ ]:
summaryDataPath = resultsPath + '/summaryData_' + dateNoTime + '.pkl'
summaryData.to_pickle(summaryDataPath)

# for excel readable summary
summaryDataPathCSV = resultsPath + '/summaryData_' + dateNoTime + '.csv'
summaryData.to_csv(summaryDataPathCSV)

#### Quick plots

In [ ]:
print(f"Available variables for summary data plotting are: {summaryData.columns}\n")
print(f"Available variables for spot data plotting are: {spotData.columns}\n")

Burst fraction

In [ ]:
var1 = "spotFrequency"

fig = plt.figure(figsize=(2.5,3))
sns.barplot(data=summaryData,y=var1,x="selection",hue="selection",palette="rocket_r")
plt.ylabel("Fraction of cells actively transcribing",fontsize=8)
plt.yticks(fontsize=8)	
plt.xlabel("")
plt.xticks(fontsize=8)
plt.title("Burst fraction",fontsize=10)
plt.legend().remove()
plt.tight_layout()
plt.show()

# EDIT BELOW TO SAVE FIGURE IF DESIRED
# figPath = resultsPath + '/burstFraction_Fig_' + dateNoTime + '.pdf'
# fig.savefig(figPath)

Mean spots per cell

In [ ]:
var1 = "meanSpotsPerCell"

fig = plt.figure(figsize=(2.5,3))
sns.barplot(data=summaryData,y=var1,x="selection",hue="selection",palette="rocket_r")
plt.ylabel("Mean transcription sites per cell",fontsize=10)
plt.yticks(fontsize=8)	
plt.xlabel("")
plt.xticks(fontsize=8)
plt.title(f"Active transcription\nsites per cell",fontsize=10)
plt.legend().remove()
plt.tight_layout()
plt.show()

# EDIT BELOW TO SAVE FIGURE IF DESIRED
# figPath = resultsPath + '/meanSpotsPerCell_Fig_' + dateNoTime + '.pdf'
# fig.savefig(figPath)

Spot intensity (approx. burst size)

In [ ]:
var1 = "targetNormalised_nc"
var2 = "medianSpotInt_nucNorm"

summaryDataToPlot = summaryData

fig = plt.figure(figsize=(2.5,3))
sns.stripplot(data=spotData,y=var1,x="selection",hue="selection",palette="rocket_r",size=2)
sns.stripplot(data=summaryDataToPlot,y=var2,x="selection",hue="selection",color="black",size=10,marker="_",linewidth=1.5)
plt.ylabel("Normalised spot intensity",fontsize=10)
plt.yticks(fontsize=8)	
plt.xlabel("")
plt.xticks(fontsize=8)
plt.title("Burst size",fontsize=10)
plt.legend().remove()
plt.tight_layout()
plt.show()

# EDIT BELOW TO SAVE FIGURE IF DESIRED
# figPath = resultsPath + '/burstSize_Fig_' + dateNoTime + '.pdf'
# fig.savefig(figPath)